# Handling Missing Data: Why It Deserves Special Attention
In this notebook, we explore why **missing data** is a crucial issue and demonstrate different strategies to handle it effectively. We'll use a sample dataset with missing values to see practical solutions in action.

## 1️⃣ Loading the Data
Let's load our dataset and take an initial look.

In [5]:
import pandas as pd

# Load the dataset
df = pd.read_csv('sample_missing_data.csv')
df.head()

,ID,Age,Salary,Department
0,1,25.0,50000.0,Sales
1,2,30.0,60000.0,IT
2,3,NaN,55000.0,HR
3,4,22.0,NaN,HR
4,5,28.0,62000.0,NaN


## 2️⃣ Identifying Missing Data
We need to see where and how much data is missing.

In [8]:
# Check for missing values
df.info()

# Visualize missing counts
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   ID          20 non-null     int64  
 1   Age         15 non-null     float64
 2   Salary      14 non-null     float64
 3   Department  16 non-null     object 
dtypes: float64(2), int64(1), object(1)
memory usage: 772.0+ bytes


ID            0
Age           5
Salary        6
Department    4
dtype: int64

## 3️⃣ Strategy 1: Listwise Deletion (Dropping Missing Data)
This method simply removes any rows with missing values. It's straightforward but can shrink your dataset a lot.

In [11]:
# Drop rows with any missing values
df_drop = df.dropna()
print(f"Original dataset size: {df.shape[0]} rows")
print(f"After dropping missing: {df_drop.shape[0]} rows")
df_drop

Original dataset size: 20 rows
After dropping missing: 7 rows


,ID,Age,Salary,Department
0,1,25.0,50000.0,Sales
1,2,30.0,60000.0,IT
7,8,40.0,61000.0,HR
10,11,27.0,57000.0,IT
13,14,24.0,59000.0,HR
15,16,33.0,65000.0,IT
19,20,29.0,61000.0,HR


## 4️⃣ Strategy 2: Mean Substitution
We replace missing numeric values with the column mean. This keeps the dataset size intact but may underestimate variability.

In [31]:
# Fill missing Age and Salary with mean
import warnings
warnings.filterwarnings('ignore')

df_mean = df.copy()
df_mean['Age'].fillna(df['Age'].mean(), inplace=True)
df_mean['Salary'].fillna(df['Salary'].mean(), inplace=True)
df_mean

,ID,Age,Salary,Department
0,1,25.000000,50000.0,Sales
1,2,30.000000,60000.0,IT
2,3,29.933333,55000.0,HR
3,4,22.000000,58500.0,HR
4,5,28.000000,62000.0,NaN
5,6,35.000000,58500.0,Sales
6,7,29.933333,52000.0,IT
7,8,40.000000,61000.0,HR
8,9,31.000000,58500.0,Sales
9,10,29.933333,58000.0,NaN


## 5️⃣ Strategy 3: Forward Fill for Categorical Columns
For categorical columns like Department, we can fill missing data using the previous (forward-fill) value.

In [29]:
import warnings
warnings.filterwarnings('ignore')
# Forward fill categorical data
df_ffill = df_mean.copy()
df_ffill['Department'].fillna(method='ffill', inplace=True)
df_ffill

,ID,Age,Salary,Department
0,1,25.000000,50000.0,Sales
1,2,30.000000,60000.0,IT
2,3,29.933333,55000.0,HR
3,4,22.000000,58500.0,HR
4,5,28.000000,62000.0,HR
5,6,35.000000,58500.0,Sales
6,7,29.933333,52000.0,IT
7,8,40.000000,61000.0,HR
8,9,31.000000,58500.0,Sales
9,10,29.933333,58000.0,Sales


## 6️⃣ Strategy 4: Multiple Imputation (Simplified Example)
Multiple Imputation means filling missing values **many times** to reflect uncertainty. Here we'll demonstrate using `SimpleImputer` with iterative strategies.

In [20]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# Using IterativeImputer for numeric columns
imputer = IterativeImputer(random_state=0)
df_iter = df.copy()
df_iter[['Age', 'Salary']] = imputer.fit_transform(df[['Age', 'Salary']])
df_iter

,ID,Age,Salary,Department
0,1,25.000000,50000.000000,Sales
1,2,30.000000,60000.000000,IT
2,3,26.007409,55000.000000,HR
3,4,22.000000,54529.551779,HR
4,5,28.000000,62000.000000,NaN
5,6,35.000000,62070.641955,Sales
6,7,23.287215,52000.000000,IT
7,8,40.000000,61000.000000,HR
8,9,31.000000,59750.306516,Sales
9,10,28.727603,58000.000000,NaN


## ✅ Summary
- **Listwise deletion** is easy but can lose a lot of data.
- **Mean substitution** is simple but may reduce variability.
- **Forward-fill** works well for ordered categories.
- **Multiple imputation** reflects uncertainty and is more robust for advanced work.

Always consider the context of your data before choosing a method!